In [ ]:
"""
Project 02: Predictive Maintenance
Step 01: EDA & Feature Engineering
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import time

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 01: EDA & Feature Engineering")
print("=" * 60)
start_time = time.time()

In [ ]:
# 1. Load Data
print("\n[1/5] Loading AI4I 2020 Predictive Maintenance dataset...")
df = pd.read_csv(os.path.join(RAW, "ai4i2020.csv"))
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"  Missing: {df.isnull().sum().sum()} total")
print(f"\n  Data types:")
print(f"  {df.dtypes.value_counts().to_dict()}")

In [ ]:
# 2. Target Engineering - Combine Binary Flags to Multiclass
print("\n[2/5] Engineering target from failure flags...")
failure_flags = ["TWF", "HDF", "PWF", "OSF", "RNF"]

In [ ]:
def get_failure_type(row):
    for f in failure_flags:
        if row[f] == 1:
            return f
    return "No Failure"

In [ ]:
df["Target"] = df.apply(get_failure_type, axis=1)

In [ ]:
print(f"\n  Class Distribution:")
class_counts = df["Target"].value_counts()
for cls, cnt in class_counts.items():
    pct = 100 * cnt / len(df)
    print(f"  {cls}: {cnt} ({pct:.1f}%)")

In [ ]:
# 3. Feature Engineering
print("\n[3/5] Engineering features...")
for i in tqdm(range(1), desc="  Feature engineering"):
    # Temperature differential (thermodynamic health indicator)
    df["Temperature Diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]

    # Power proxy (kinetic energy indicator)
    df["Power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"] * 0.10472

In [ ]:
print(f"  Engineered: Temperature Diff, Power")

In [ ]:
# 4. Subset & Clean Column Names (XGBoost hates brackets)
print("\n[4/5] Cleaning column names and selecting features...")
features_map = {
    "Air temperature [K]": "Air temperature",
    "Process temperature [K]": "Process temperature",
    "Temperature Diff": "Temperature Diff",
    "Rotational speed [rpm]": "Rotational speed",
    "Torque [Nm]": "Torque",
    "Power": "Power",
    "Tool wear [min]": "Tool wear",
    "Target": "Target",
}

In [ ]:
processed_df = df[list(features_map.keys())].rename(columns=features_map)
print(f"  Final features: {list(processed_df.columns)}")

In [ ]:
# 5. Save Processed Data
out_path = os.path.join(PROCESSED, "processed_data.csv")
processed_df.to_csv(out_path, index=False)
print(f"\n  Saved to: {out_path}")
print(f"  Shape: {processed_df.shape}")

In [ ]:
# 6. Generate Charts
print("\n[5/5] Generating EDA charts...")

In [ ]:
# Chart 1: Class Distribution
for i in tqdm(range(1), desc="  Class distribution chart"):
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#3498db", "#1abc9c"]
    class_counts.plot(
        kind="bar", ax=ax, color=colors[: len(class_counts)], edgecolor="black"
    )
    ax.set_title("Failure Type Distribution (96.5% No Failure Imbalance)", fontsize=14)
    ax.set_ylabel("Count")
    ax.set_xlabel("Failure Type")
    for j, (cls, cnt) in enumerate(class_counts.items()):
        ax.text(
            j, cnt + 50, f"{cnt}\n({100 * cnt / len(df):.1f}%)", ha="center", fontsize=9
        )
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "class_distribution.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 2: Feature Distributions by Failure Type
for i in tqdm(range(1), desc="  Feature distributions chart"):
    numeric_features = [
        "Air temperature",
        "Process temperature",
        "Temperature Diff",
        "Rotational speed",
        "Torque",
        "Power",
        "Tool wear",
    ]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    for idx, feat in enumerate(numeric_features):
        if idx < len(axes):
            for target in processed_df["Target"].unique():
                subset = processed_df[processed_df["Target"] == target]
                axes[idx].hist(subset[feat], alpha=0.5, label=target, bins=30)
            axes[idx].set_title(feat, fontsize=10)
            axes[idx].tick_params(labelsize=8)
    if len(numeric_features) < len(axes):
        axes[-1].set_visible(False)
    plt.suptitle("Feature Distributions by Failure Type", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "feature_distributions.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 3: Correlation Heatmap
for i in tqdm(range(1), desc="  Correlation heatmap"):
    fig, ax = plt.subplots(figsize=(10, 8))
    numeric_cols = processed_df.select_dtypes(include=[np.number]).columns
    corr = processed_df[numeric_cols].corr()
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        ax=ax,
        square=True,
        linewidths=0.5,
    )
    ax.set_title("Feature Correlation Heatmap", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "correlation_heatmap.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 4: Box Plots by Failure Type
for i in tqdm(range(1), desc="  Box plots chart"):
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    for idx, feat in enumerate(numeric_features):
        if idx < len(axes):
            processed_df.boxplot(column=feat, by="Target", ax=axes[idx])
            axes[idx].set_title(feat, fontsize=10)
            axes[idx].set_xlabel("")
            axes[idx].tick_params(labelsize=7, axis="x", rotation=45)
    if len(numeric_features) < len(axes):
        axes[-1].set_visible(False)
    plt.suptitle("Feature Ranges by Failure Type", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "boxplots_by_failure.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 5: Temperature Diff vs Power scatter
for i in tqdm(range(1), desc="  Scatter plot chart"):
    fig, ax = plt.subplots(figsize=(10, 7))
    failure_types = processed_df["Target"].unique()
    colors_scatter = ["#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#3498db", "#1abc9c"]
    for idx, ft in enumerate(failure_types):
        subset = processed_df[processed_df["Target"] == ft]
        ax.scatter(
            subset["Temperature Diff"],
            subset["Power"],
            alpha=0.4,
            s=10,
            label=ft,
            color=colors_scatter[idx % len(colors_scatter)],
        )
    ax.set_xlabel("Temperature Diff (K)")
    ax.set_ylabel("Power (W)")
    ax.set_title("Temperature Diff vs Power by Failure Type", fontsize=14)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "tempdiff_vs_power.png"), dpi=150)
    plt.close()

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 01 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")